<a href="https://colab.research.google.com/github/Jbrr2021/gcp-finops-python-automation/blob/main/uditoria_regioes_cloud_pandas_startswith_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Pipeline de Auditoria de Compliance e Validação Geográfica

## 🗂️ Conceito Técnico: Integração Python-BigQuery com Validação de Strings (`.str.startswith`) no Pandas

### 💼 Contexto de Negócio (Pedido do Gestor):
> *"Para evitar custos adicionais com impostos de importação e moedas estrangeiras, a diretoria exige que recursos críticos rodem apenas em regiões homologadas (EUA). Crie um script em Python que consuma os dados brutos do BigQuery e utilize o Pandas para auditar a coluna de localização. Se a região não iniciar com o padrão 'us-', aplique o carimbo '⚠️ ATENÇÃO: Região Estrangeira'; caso contrário, rotule como '✅ Região Homologada', exportando a planilha tratada automaticamente para o Excel do time de compliance."*

---

### ⚙️ Engenharia da Solução (O que este código faz):
1. **Autenticação e Conexão:** Realiza o handshake seguro via `auth.authenticate_user()` e instancia o cliente oficial do BigQuery injetando credenciais ativas do GCP [financeira].
2. **Ingestão via Notação de Ponto:** Executa uma query analítica extraindo o ID do projeto, descrição do serviço, custos e faz a navegação no objeto aninhado `location.location` para capturar a região [financeira].
3. **Conversão Estruturada:** Transforma o retorno de Big Data do banco diretamente em um `DataFrame` bidimensional do Pandas utilizando o método `.to_dataframe()` [financeira].
4. **Transformação Vetorizada (Lambda):** Cria a nova coluna de compliance aplicando a função `.str.startswith('us-')` para mapear os data centers norte-americanos de forma performática [financeira].
5. **Higienização de Formato:** Trata o arquivo de saída configurando delimitador nacional brasileiro (`sep=';'`) e injeção de BOM (`encoding='utf-8-sig'`) para garantir abertura perfeita de acentos e emojis diretamente no Microsoft Excel [financeira].
6. **Carga Automatizada:** Dispara o gatilho de download (`files.download`) do relatório físico de auditoria de forma 100% autônoma [financeira].


In [10]:
# =================================================================================
# PIPELINES DE ENGENHARIA DE DADOS / AUDITORIA DE RECURSOS E COMPLIANCE
# CONCEITO: Uso do Pandas (.str.startswith e Lambda) para Validação de Strings
#
# CONTEXTO DE NEGÓCIO (Pedido do Gestor):
# "Para evitar custos adicionais com impostos de importação e moedas estrangeiras,
# a diretoria exige que recursos críticos rodem apenas em regiões homologadas (EUA).
# Crie um script em Python que consuma os dados brutos do BigQuery e utilize o Pandas
# para auditar a coluna de localização. Se a região não iniciar com o padrão 'us-',
# aplique o carimbo '⚠️ ATENÇÃO: Região Estrangeira'; caso contrário, rotule como
# '✅ Região Homologada', exportando a planilha tratada para o time de compliance."
# =================================================================================


from google.cloud.bigquery import query
from google.cloud import bigquery
import pandas as pd
import google.auth #Bliblioteca para capturar minhas credencias
from google.colab import files
from google.colab import auth # Importar o módulo de autenticação

# Adicionar autenticação para usuário no Colab
print("Authenticating use...")
auth.authenticate_user()
print("User authenticated.")

# 1. CAPTURA DOS TOKENS: Puxa explicitamente a autenticação realizada na célula 1
crendenciais, projeto_padrao = google.auth.default()

# 2. Inicializando o cliente injetando as suas credencias de usuário logado.
client = bigquery.Client(credentials=crendenciais, project='gen-lang-client-0862909089')

# 3. Definir a query dinâmica de subquery
query = """
SELECT
    project.id AS projeto_id,
    service.description AS servico_descricao,
    location.location AS regiao_codigo,
    cost AS custo_bruto
FROM
    `meu_finops.custos_nuvem_bruto_gcp`
ORDER BY
    custo_bruto DESC;
  """
print("Buscando dados no BigQuery...")
# Executando a quary e convertento para DataFrame do Pandas
df_custo = client.query(query).to_dataframe()

# 4. Criando a nova coluna de auditoria baseada no início do texto da região
df_custo['auditoria_regiao'] = df_custo['regiao_codigo'].apply(
    lambda x: '✅ Região Homologada' if x.startswith('us-') else '⚠️ ATENÇÃO: Região Estrangeira'
)

# 5. Exibir a tabla do colab para validar
print(df_custo)

# 6. Salvar e fazer dawnload
nome_arquivo = "relatorio_auditoria_regioes.csv"
df_custo.to_csv(nome_arquivo, index=False, sep=';', encoding='utf-8-sig')
files.download(nome_arquivo)


Authenticating use...
User authenticated.
Buscando dados no BigQuery...
          projeto_id servico_descricao       regiao_codigo  custo_bruto  \
0  ipnet-vendas-prod    Compute Engine  southamerica-east1        450.0   
1    ipnet-data-lake          BigQuery         us-central1        320.5   
2  ipnet-vendas-prod     Cloud Storage            us-east1         25.0   

                 auditoria_regiao  
0  ⚠️ ATENÇÃO: Região Estrangeira  
1             ✅ Região Homologada  
2             ✅ Região Homologada  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>